In [0]:

df = spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2020")
df.printSchema()
display(df.limit(10))

print("Total de filas:", df.count())
print("Valores únicos en 'sex':")
df.select("sex").distinct().show()
print("Valores nulos por columna:")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
from pyspark.sql import functions as F

df = spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2020")

print("Valores nulos por columna:")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
from pyspark.sql import functions as F

# Leer desde Bronce
df_bronze = spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2020")

df_silver = (
    df_bronze
    
    # Regla 5 — correct_data_types: convertir a tipos correctos
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("number", F.col("number").cast("int"))  # es un conteo de muertes, debe ser entero
    .withColumn("percentage_of_cause_specific_deaths_out_of_total_deaths", 
                F.col("percentage_of_cause_specific_deaths_out_of_total_deaths").cast("double"))
    .withColumn("age_standardized_death_rate_per_100_000_standard_population", 
                F.col("age_standardized_death_rate_per_100_000_standard_population").cast("double"))
    .withColumn("death_rate_per_100_000_population", 
                F.col("death_rate_per_100_000_population").cast("double"))
    
    # Regla 6 — standardize_categorical_values: estandarizar sexo a código corto
    .withColumn("sex", 
                F.when(F.col("sex") == "Male", "M")
                 .when(F.col("sex") == "Female", "F")
                 .when(F.col("sex") == "All", "TODOS")
                 .otherwise(F.col("sex")))
    
    # Regla 6 — limpiar age_group: quitar corchetes de "[All]" -> "All"
    .withColumn("age_group", F.regexp_replace(F.col("age_group"), r"[\[\]]", ""))
    
    # Regla 7 — semantic_validation: eliminar filas con conteos negativos (imposible)
    .filter(F.col("number") >= 0)
    
    # Agregar país de origen (necesario para cuando consolidemos en Gold con otras fuentes)
    .withColumn("pais", F.lit("Costa Rica"))
    .withColumn("codigo_pais", F.lit("CRI"))
    
    # Metadata de control de Silver
    .withColumn("silver_processed_at", F.current_timestamp())
)

# Regla 3 — remove_duplicate_rows
filas_antes = df_silver.count()
df_silver = df_silver.dropDuplicates()
filas_despues = df_silver.count()
print(f"Filas antes de quitar duplicados: {filas_antes}")
print(f"Filas después de quitar duplicados: {filas_despues}")
print(f"Duplicados eliminados: {filas_antes - filas_despues}")

display(df_silver.limit(10))
df_silver.printSchema()

In [0]:
#Guardamos en esquema silver la primera tabla
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("covid19.silver.mortalidad_categorias_costa_rica_2020")

print("✓ Tabla Silver creada exitosamente")

# Verificar
spark.sql("SELECT COUNT(*) as total_filas FROM covid19.silver.mortalidad_categorias_costa_rica_2020").show()

In [0]:
from pyspark.sql import functions as F

# Leer desde Bronce
df_bronze = spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2021")

df_silver = (
    df_bronze
    
    # Regla 5 — correct_data_types: convertir a tipos correctos
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("number", F.col("number").cast("int"))  # es un conteo de muertes, debe ser entero
    .withColumn("percentage_of_cause_specific_deaths_out_of_total_deaths", 
                F.col("percentage_of_cause_specific_deaths_out_of_total_deaths").cast("double"))
    .withColumn("age_standardized_death_rate_per_100_000_standard_population", 
                F.col("age_standardized_death_rate_per_100_000_standard_population").cast("double"))
    .withColumn("death_rate_per_100_000_population", 
                F.col("death_rate_per_100_000_population").cast("double"))
    
    # Regla 6 — standardize_categorical_values: estandarizar sexo a código corto
    .withColumn("sex", 
                F.when(F.col("sex") == "Male", "M")
                 .when(F.col("sex") == "Female", "F")
                 .when(F.col("sex") == "All", "TODOS")
                 .otherwise(F.col("sex")))
    
    # Regla 6 — limpiar age_group: quitar corchetes de "[All]" -> "All"
    .withColumn("age_group", F.regexp_replace(F.col("age_group"), r"[\[\]]", ""))
    
    # Regla 7 — semantic_validation: eliminar filas con conteos negativos (imposible)
    .filter(F.col("number") >= 0)
    
    # Agregar país de origen (necesario para cuando consolidemos en Gold con otras fuentes)
    .withColumn("pais", F.lit("Costa Rica"))
    .withColumn("codigo_pais", F.lit("CRI"))
    
    # Metadata de control de Silver
    .withColumn("silver_processed_at", F.current_timestamp())
)

# Regla 3 — remove_duplicate_rows
filas_antes = df_silver.count()
df_silver = df_silver.dropDuplicates()
filas_despues = df_silver.count()
print(f"Filas antes de quitar duplicados: {filas_antes}")
print(f"Filas después de quitar duplicados: {filas_despues}")
print(f"Duplicados eliminados: {filas_antes - filas_despues}")

display(df_silver.limit(10))
df_silver.printSchema()